# Privacy Auditing

Privacy accounting gives *theoretical* upper bounds on privacy loss.
Privacy auditing gives *empirical* lower bounds by running a
membership inference attack. If the audited epsilon exceeds the
theoretical bound, there is likely a bug.

This notebook runs the full auditing workflow end-to-end:
partition canaries, train with DP-SGD, score, and estimate epsilon.

We use **random labels** — a standard stress-test that forces the
model to memorize training data, producing a strong membership
inference signal.

**Prerequisites:** Familiarity with DP-SGD (see the
[DP-SGD training tutorial](dp_sgd_training.ipynb)).

**Components:** `coin_flip`, `loss_scores`, `one_run`,
`OneRunEstimate`.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import opaque.accounting as acc
import opaque.auditing as auditing
from opaque.clipping import clipped_grad
from opaque.dpsgd.noise import gaussian_noise
from opaque.functional import make_functional
from opaque.random import key

torch.manual_seed(42)

## Dataset and model

Random labels cannot be learned by generalisation — the model
must memorise. A small MLP has enough capacity to overfit,
producing a clear gap between in-member and out-member losses.

In [ ]:
n_samples, n_features = 500, 10
X = torch.randn(n_samples, n_features)
y = torch.randint(0, 2, (n_samples,)).float()  # random labels
dataset = TensorDataset(X, y)

model = nn.Sequential(
    nn.Linear(n_features, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
)
fmodel, params = make_functional(model)


def loss_fn(params, x, y):
    logit = fmodel(params, x.unsqueeze(0)).squeeze()
    return F.binary_cross_entropy_with_logits(logit, y)

## Step 1 — Partition canaries

`coin_flip()` randomly selects canary examples and flips a fair
coin for each: *in* (included in training) or *out* (held out).
Non-canary examples are always included.

In [ ]:
num_canaries = n_samples  # audit every example
cf = auditing.coin_flip(dataset, num_canaries=num_canaries, key=key(42))
train_data = cf.train_subset(dataset)

print(f"Dataset:      {n_samples} examples")
print(
    f"Canaries:     {num_canaries} ({len(cf.in_indices)} in, {len(cf.out_indices)} out)"
)
print(f"Training set: {len(train_data)} examples")

## Step 2 — Set up DP-SGD and compute reference scores

Before training we record each canary’s loss on the
untrained model. The final membership scores will be the
loss *reduction* from training (Steinke et al., Algorithm 3).

In [ ]:
batch_size = 64
clipping_norm = 1.0
noise_multiplier = 0.5
n_epochs = 200
lr = 0.5

grad_fn, clip_state = clipped_grad(
    loss_fn,
    argnums=0,
    batch_argnums=(1, 2),
    clipping_norm=clipping_norm,
)
noise_fn, noise_state = gaussian_noise(
    noise_multiplier=noise_multiplier,
    key=key(0),
)

# Score canaries on the untrained model
canary_loader = DataLoader(
    cf.canary_subset(dataset),
    batch_size=batch_size,
    shuffle=False,
)
ref_scores = auditing.loss_scores(
    loss_fn,
    params,
    batch_argnums=(1, 2),
    dataloader=canary_loader,
)
print(f"Reference scores: mean={ref_scores.mean():.4f}, std={ref_scores.std():.4f}")

## Step 3 — Train

In [ ]:
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

for epoch in range(n_epochs):
    for xb, yb in train_loader:
        bs = xb.shape[0]
        grads, clip_state = grad_fn(params, xb, yb, state=clip_state)
        noisy_grads, noise_state = noise_fn(grads, noise_state)
        params = tuple(
            p - (lr / bs) * g for p, g in zip(params, noisy_grads.pytree)
        )

    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            logits = fmodel(params, X[:200]).squeeze()
            accuracy = (logits > 0).float().eq(y[:200]).float().mean()
        print(f"Epoch {epoch + 1:3d}  accuracy={accuracy:.1%}")

## Step 4 — Score canaries and estimate epsilon

`loss_scores()` computes per-canary membership scores (negated loss)
using `vmap`. Passing `reference_scores` subtracts the untrained
baseline so the score measures *loss reduction* from training.

`one_run()` then performs the likelihood-ratio test from
Steinke et al. (2023) to estimate epsilon.

In [ ]:
scores = auditing.loss_scores(
    loss_fn,
    params,
    batch_argnums=(1, 2),
    dataloader=canary_loader,
    reference_scores=ref_scores,
)

estimate = auditing.one_run(scores, coin_flip=cf)

## Results

Compare the empirical epsilon from the audit against the
theoretical bound from privacy accounting.

In [ ]:
# Theoretical bound from privacy accounting
sample_rate = batch_size / len(train_data)
total_steps = n_epochs * (len(train_data) // batch_size)
process = acc.poisson(acc.gaussian(noise_multiplier), sample_rate) * total_steps
theoretical_eps = process.epsilon_at(1e-5)

delta = 1e-5
audited_eps = estimate.epsilon_at(delta=delta)

print(f"Audited epsilon:      {audited_eps:.2f}")
print(f"Theoretical epsilon:  {theoretical_eps:.2f}")
if audited_eps <= theoretical_eps:
    print("OK: audited epsilon is within the theoretical bound.")
else:
    print("WARNING: audited epsilon exceeds theoretical bound.")

The audited epsilon should be *below* the theoretical bound.

**Why is the gap so large?** Privacy auditing is a *black-box*
method: it can only detect leakage that its specific membership
inference attack (here, loss thresholding) can exploit. The
theoretical bound, by contrast, holds against *all* possible
adversaries, including ones far more powerful than any single
MIA. The audited epsilon is therefore a *lower bound* on the
true privacy leakage — it tells you "at least this much leaks,"
but the actual leakage could be anywhere between the audited and
theoretical values. A stronger attack would narrow the gap.

## Attack metrics

AUC measures how well an adversary can distinguish in-members
from out-members. Values near 0.5 mean the model leaks little.

In [ ]:
print(f"AUC:        {estimate.auc():.3f}")
print(f"β@α=0.01: {estimate.beta_at(alpha=0.01):.3f}")

ci = estimate.auc(confidence=0.95, num_samples=1000, key=key(42))
print(f"AUC 95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]")

## Score distributions

Visualise how in-member and out-member scores overlap.
More overlap means better privacy.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(estimate.in_scores, bins=30, alpha=0.6, label="In", density=True)
ax.hist(estimate.out_scores, bins=30, alpha=0.6, label="Out", density=True)
ax.set_xlabel("Score")
ax.set_ylabel("Density")
ax.set_title("Canary score distributions")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

The auditing workflow is three steps after preparing the data:

1. **Partition** — `auditing.coin_flip(dataset, num_canaries=..., key=...)`
2. **Train** — on `cf.train_subset(dataset)` with normal DP-SGD
3. **Score & estimate** — `auditing.loss_scores(...)` then
   `auditing.one_run(scores, coin_flip=cf)`

See `examples/train_causal_lm.py --audit` for a complete LLM example.